# Beginner E1: Librosa features + Random Forest

This notebook teaches a second traditional machine-learning baseline for E1. It uses the same handcrafted audio features and speaker-disjoint evaluation as the beginner SVM notebook, but trains a **Random Forest**.

You will learn how to:

1. extract MFCC, pitch, spectral, zero-crossing-rate and energy features with Librosa;
2. keep speakers separate between training and testing;
3. train a collection of decision trees;
4. evaluate F1, balanced accuracy and the confusion matrix; and
5. inspect which acoustic features the forest used most.

> This experiment distinguishes the supplied datasets. It must not be used to infer ethnicity, nationality or identity.

In [ ]:
# Install the libraries used in this notebook.
%pip -q install "librosa>=0.10.2" scikit-learn seaborn joblib soundfile

## 1. Arrange the Google Drive project

Manually upload the audio under one `Singlish` project folder. The first folder below each class identifies the speaker:

```text
Singlish/
├── data/
│   └── audio/
│       ├── singapore/
│       │   ├── speaker_001/
│       │   │   ├── clip_01.wav
│       │   │   └── clip_02.wav
│       │   └── speaker_002/
│       │       └── clip_01.wav
│       └── non_singapore/
│           ├── speaker_101/
│           │   └── clip_01.wav
│           └── speaker_102/
│               └── clip_01.wav
```

Only `data/audio` needs to exist before the run. The notebook automatically creates `Singlish/outputs/e1` for its results.

Each class needs at least two speakers; five or more is preferable. Speaker folders prevent one person's clips from appearing in both training and testing.

In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# Configure only this path.
PROJECT_ROOT = Path('/content/drive/MyDrive/Singlish')
DATA_ROOT = PROJECT_ROOT / 'data' / 'audio'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'e1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
SAMPLE_RATE = 16_000
MAX_SECONDS = 6

print('Project folder:', PROJECT_ROOT)
print('Looking for audio in:', DATA_ROOT)
print('Saving results to:', OUTPUT_DIR)

In [ ]:
import json
from collections import Counter

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold

np.random.seed(RANDOM_SEED)
print('Librosa version:', librosa.__version__)

## 2. Find and label the recordings

Singapore speech receives label `1`; non-Singapore speech receives label `0`. The source-prefixed speaker group prevents an ID collision between corpora.

In [ ]:
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
CLASS_LABELS = {'non_singapore': 0, 'singapore': 1}


def find_recordings(data_root):
    rows = []
    for class_name, label in CLASS_LABELS.items():
        class_dir = data_root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f'Missing class directory: {class_dir}')
        for path in sorted(class_dir.rglob('*')):
            if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
                continue
            parts = path.relative_to(class_dir).parts
            if len(parts) < 2:
                raise ValueError(f'Put this clip inside a speaker folder: {path}')
            group_id = f'{class_name}::{parts[0]}'
            rows.append((path, label, group_id))
    if not rows:
        raise RuntimeError(f'No supported audio files found below {data_root}')
    return rows


recordings = find_recordings(DATA_ROOT)
print('Recordings:', len(recordings))
print('Labels:', Counter(label for _, label, _ in recordings))
for class_name, label in CLASS_LABELS.items():
    speakers = {group for _, item_label, group in recordings if item_label == label}
    print(f'{class_name} speakers: {len(speakers)}')

## 3. Extract 39 acoustic features

An audio clip has a variable number of waveform samples, while a classifier expects a fixed number of inputs. Librosa measures acoustic properties frame by frame; their means and standard deviations turn every clip into 39 values.

The vector contains 13 MFCC means, 13 MFCC standard deviations, three pitch statistics, and mean/standard deviation for spectral centroid, bandwidth, roll-off, zero-crossing rate and RMS energy.

In [ ]:
N_MFCC = 13
N_FFT = 1024
HOP_LENGTH = 256

FEATURE_NAMES = (
    [f'mfcc_{i + 1}_mean' for i in range(N_MFCC)]
    + [f'mfcc_{i + 1}_std' for i in range(N_MFCC)]
    + ['f0_mean', 'f0_std', 'voiced_ratio']
    + [
        f'{family}_{stat}'
        for family in ('spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'zcr', 'rms')
        for stat in ('mean', 'std')
    ]
)
assert len(FEATURE_NAMES) == 39


def load_and_prepare_audio(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    if y.size == 0:
        raise ValueError(f'No audible signal: {path}')
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    y = librosa.util.fix_length(y, size=max(len(y), SAMPLE_RATE))
    return y[:SAMPLE_RATE * MAX_SECONDS].astype(np.float32)


def mean_and_std(feature):
    return np.concatenate([np.mean(feature, axis=1), np.std(feature, axis=1)])


def extract_features(path):
    y = load_and_prepare_audio(path)
    mfcc = librosa.feature.mfcc(
        y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
    )
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    f0 = librosa.yin(
        y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'),
        sr=SAMPLE_RATE, frame_length=2048, hop_length=HOP_LENGTH,
    )
    energy = rms.ravel()[:len(f0)]
    voiced = np.isfinite(f0) & (energy > max(float(np.median(energy)) * 0.1, 1e-4))
    voiced_f0 = f0[voiced]
    pitch_stats = [
        float(np.mean(voiced_f0)) if voiced_f0.size else 0.0,
        float(np.std(voiced_f0)) if voiced_f0.size else 0.0,
        float(np.mean(voiced)),
    ]
    time_features = [
        librosa.feature.spectral_centroid(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.spectral_bandwidth(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.spectral_rolloff(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH),
        rms,
    ]
    vector = np.concatenate([mean_and_std(mfcc), pitch_stats] + [mean_and_std(x) for x in time_features])
    vector = np.nan_to_num(vector, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    assert vector.shape == (len(FEATURE_NAMES),)
    return vector

In [ ]:
feature_rows, kept_rows = [], []
for number, row in enumerate(recordings, start=1):
    path, label, group_id = row
    try:
        feature_rows.append(extract_features(path))
        kept_rows.append(row)
    except Exception as error:
        print(f'Skipping {path.name}: {error}')
    if number % 25 == 0 or number == len(recordings):
        print(f'Processed {number}/{len(recordings)}')

X = np.vstack(feature_rows)
y = np.asarray([label for _, label, _ in kept_rows], dtype=np.int64)
groups = np.asarray([group for _, _, group in kept_rows])
paths = np.asarray([str(path) for path, _, _ in kept_rows])
assert X.shape[1] == len(FEATURE_NAMES) and np.isfinite(X).all()
print('Feature matrix shape:', X.shape)

## 4. Make a speaker-disjoint split

A clip-level random split can let the forest recognize voices it already heard. `StratifiedGroupKFold` keeps every speaker entirely in training or testing while preserving both labels as closely as possible. The same seed and sorted file order as the beginner SVM notebook produce the same split.

In [ ]:
speaker_counts = {label: len(set(groups[y == label])) for label in sorted(np.unique(y))}
if set(speaker_counts) != {0, 1}:
    raise ValueError('Both classes must remain after feature extraction')
n_splits = min(5, min(speaker_counts.values()))
if n_splits < 2:
    raise ValueError('Each class needs audio from at least two speakers')

splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(X, y, groups))
train_speakers = set(groups[train_idx])
test_speakers = set(groups[test_idx])
assert not (train_speakers & test_speakers)
assert set(np.unique(y[train_idx])) == {0, 1}
assert set(np.unique(y[test_idx])) == {0, 1}

print('Training clips:', len(train_idx), '| labels:', Counter(y[train_idx]))
print('Test clips:    ', len(test_idx), '| labels:', Counter(y[test_idx]))
print('Speaker overlap:', len(train_speakers & test_speakers))

## 5. Train the Random Forest

A decision tree repeatedly splits the feature space into smaller regions. A Random Forest trains many slightly different trees on bootstrapped samples and lets them vote, which is usually more stable than one tree.

Unlike an SVM, a Random Forest does **not** require feature scaling: a tree compares one feature with a threshold at a time. `min_samples_leaf=2` adds mild regularization, and `class_weight='balanced'` protects against small class-count differences.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    min_samples_leaf=2,
    class_weight='balanced',
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
model.fit(X[train_idx], y[train_idx])
print('Training complete.')
print(f'Out-of-bag accuracy estimate: {model.oob_score_:.3f}')

The out-of-bag score uses training examples omitted from each tree's bootstrap sample. It is a useful internal check, but the speaker-disjoint test set remains the primary evaluation because out-of-bag samples can still include other clips from the same speaker.

## 6. Evaluate unseen speakers

Balanced accuracy gives both classes equal importance. Macro F1 calculates F1 separately for both classes and averages them. The confusion matrix shows the exact types of mistakes; rows are true labels and columns are predictions.

In [ ]:
predictions = model.predict(X[test_idx])
matrix = confusion_matrix(y[test_idx], predictions, labels=[0, 1])
metrics = {
    'accuracy': float(accuracy_score(y[test_idx], predictions)),
    'balanced_accuracy': float(balanced_accuracy_score(y[test_idx], predictions)),
    'macro_f1': float(f1_score(y[test_idx], predictions, average='macro')),
    'oob_accuracy': float(model.oob_score_),
    'confusion_matrix': matrix.tolist(),
}
print(json.dumps(metrics, indent=2))
print('\nDetailed report:\n')
print(classification_report(
    y[test_idx], predictions, labels=[0, 1],
    target_names=['Non-Singapore', 'Singapore'], digits=3, zero_division=0,
))

In [ ]:
sns.set_theme(style='white')
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    matrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax,
    xticklabels=['Non-SG', 'Singapore'], yticklabels=['Non-SG', 'Singapore'],
)
ax.set(title='Beginner E1 Random Forest test', xlabel='Predicted label', ylabel='True label')
fig.tight_layout()
confusion_path = OUTPUT_DIR / 'beginner_random_forest_confusion_matrix.png'
fig.savefig(confusion_path, dpi=180, bbox_inches='tight')
plt.show()

## 7. Inspect feature importance

The forest records how much its splits reduced impurity for each input. This is useful for exploration, but importance does not prove that a feature causes an accent prediction. Correlated features can also share or distort importance.

In [ ]:
importance_order = np.argsort(model.feature_importances_)[::-1]
top_count = 15
top_indices = importance_order[:top_count][::-1]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    [FEATURE_NAMES[index] for index in top_indices],
    model.feature_importances_[top_indices],
    color='seagreen',
)
ax.set(title='Top Random Forest feature importances', xlabel='Mean decrease in impurity')
fig.tight_layout()
importance_path = OUTPUT_DIR / 'beginner_random_forest_feature_importance.png'
fig.savefig(importance_path, dpi=180, bbox_inches='tight')
plt.show()

print('Five most important features:')
for index in importance_order[:5]:
    print(f'  {FEATURE_NAMES[index]}: {model.feature_importances_[index]:.4f}')

## 8. Save the model and experiment

Everything is saved inside `Singlish/outputs/e1` in Google Drive. The feature cache contains the exact train/test indices so another classifier can be compared on the same samples.

In [ ]:
model_path = OUTPUT_DIR / 'beginner_librosa_random_forest.joblib'
cache_path = OUTPUT_DIR / 'beginner_random_forest_features.npz'
report_path = OUTPUT_DIR / 'beginner_random_forest_report.json'

joblib.dump({'model': model, 'feature_names': FEATURE_NAMES}, model_path)
np.savez_compressed(
    cache_path, features=X, labels=y, groups=groups, paths=paths,
    feature_names=np.asarray(FEATURE_NAMES),
    train_indices=train_idx, test_indices=test_idx,
)
report = {
    'model': 'RandomForestClassifier',
    'parameters': {
        'n_estimators': 300, 'max_features': 'sqrt',
        'min_samples_leaf': 2, 'class_weight': 'balanced',
    },
    'label_meaning': {'0': 'non_singapore', '1': 'singapore'},
    'feature_count': len(FEATURE_NAMES),
    'train_samples': int(len(train_idx)),
    'test_samples': int(len(test_idx)),
    'train_speakers': int(len(train_speakers)),
    'test_speakers': int(len(test_speakers)),
    'speaker_overlap': 0,
    'metrics': metrics,
    'top_features': [
        {'name': FEATURE_NAMES[index], 'importance': float(model.feature_importances_[index])}
        for index in importance_order[:15]
    ],
    'caution': 'Corpus and recording differences may be learned instead of accent.',
}
report_path.write_text(json.dumps(report, indent=2) + '\n')

print('Saved:')
for path in (model_path, cache_path, report_path, confusion_path, importance_path):
    print(' -', path)

## 9. Try one recording

The helper below applies the same feature extraction and trained forest to one file. Treat the probability as a model score, not a definitive statement about a person.

In [ ]:
def predict_recording(audio_path):
    features = extract_features(Path(audio_path)).reshape(1, -1)
    probability = float(model.predict_proba(features)[0, 1])
    predicted_name = 'Singapore' if probability >= 0.5 else 'Non-Singapore'
    print(f'Prediction: {predicted_name}')
    print(f'Singapore-class probability: {probability:.1%}')
    return probability

# Example — replace this with one of your audio paths:
# predict_recording(DATA_ROOT / 'singapore' / 'speaker_001' / 'clip_01.wav')

## What to compare with the SVM

Compare held-out macro F1, balanced accuracy and both confusion matrices. Because the notebooks use the same sorted files, seed, features and split method, their results are directly comparable when neither notebook skips a file.

Do not select the better model from repeated test-set experiments. The full E1 workflow should use a separate validation partition for model selection. Also investigate corpus, microphone and prompt differences if either model obtains an unusually high score.